In [1]:
import json
import pandas as pd
import numpy as np

from pprint import pprint
from collections import Counter, defaultdict

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [5]:
candidates = []

with open("../data/candidates.jsonl", "r", encoding = "utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

print(f"Loaded {len(candidates)} candidates")

Loaded 100000 candidates


In [6]:
pprint(candidates[0])

{'candidate_id': 'CAND_0000001',
 'career_history': [{'company': 'Mindtree',
                     'company_size': '10001+',
                     'description': 'Implemented streaming data pipelines on '
                                    'Kafka and Spark Streaming for a real-time '
                                    'user-activity processing platform. '
                                    'Designed the schema-registry integration, '
                                    'the watermark/state management approach, '
                                    'and the deduplication logic for '
                                    'late-arriving events. Worked closely with '
                                    'the data science team to make sure '
                                    'feature pipelines aligned with what their '
                                    'models needed. Most of my career has been '
                                    'data engineering, with some adjacent ML '
              

In [7]:
print(type(candidates))
print(type(candidates[0]))
print(candidates[0].keys())

<class 'list'>
<class 'dict'>
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [10]:
profiles = []
for candidate in candidates:
    profile = candidate["profile"]  

    profiles.append({
        "candidate_id" : candidate["candidate_id"],
        **profile
    })

profiles_df = pd.DataFrame(profiles)
profiles_df.head()

,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,current_industry
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud","Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,IT Services
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,"Professional with 12.5+ years of experience. My professional background is in marketing manager — I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.","Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,IT Services
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,"Professional with 1.1+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",Austin,USA,1.1,Customer Support,TCS,10001+,IT Services
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,"Professional with 3.8+ years of experience. My professional background is in marketing manager — I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,Paper Products
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,"Professional with 11.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.","Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,Manufacturing


In [11]:
profiles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   candidate_id          100000 non-null  object 
 1   anonymized_name       100000 non-null  object 
 2   headline              100000 non-null  object 
 3   summary               100000 non-null  object 
 4   location              100000 non-null  object 
 5   country               100000 non-null  object 
 6   years_of_experience   100000 non-null  float64
 7   current_title         100000 non-null  object 
 8   current_company       100000 non-null  object 
 9   current_company_size  100000 non-null  object 
 10  current_industry      100000 non-null  object 
dtypes: float64(1), object(10)
memory usage: 8.4+ MB


In [12]:
profiles_df["years_of_experience"].describe()

count    100000.000000
mean          7.166319
std           3.824551
min           1.000000
25%           3.900000
50%           6.800000
75%           9.900000
max          16.900000
Name: years_of_experience, dtype: float64

In [13]:
profiles_df.sort_values(
    "years_of_experience", ascending = False
).head()

,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,current_industry
55991,CAND_0055992,Deepak Bansal,"AI Engineer | Search, Ranking & Retrieval","Machine learning engineer with 6.8 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I care a lot about evaluation rigor — too many teams ship models without offline benchmarks they trust. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.",Sydney,Australia,16.9,AI Engineer,CRED,1001-5000,Fintech
91533,CAND_0091534,Dhruv Dutta,"AI Engineer | ML, NLP, Recommendation Systems","Machine learning engineer with 7.2 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. Along the way I've gotten comfortable with the operational side — A/B testing, drift monitoring, retraining schedules. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.","Gurgaon, Haryana",India,16.6,AI Engineer,Flipkart,10001+,E-commerce
71114,CAND_0071115,Tanya Shah,"Recommendation Systems Engineer | ML, NLP, Recommendation Systems","Machine learning engineer with 5.8 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, my main project for the last 18 months has been the recommendation system that powers our discovery feed. I care a lot about evaluation rigor — too many teams ship models without offline benchmarks they trust. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.","Ahmedabad, Gujarat",India,16.5,Recommendation Systems Engineer,Meta,10001+,Internet
39753,CAND_0039754,Mira Banerjee,Senior Applied Scientist | Building AI-native search & ranking systems,"Senior AI engineer with 8.3 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I rebuilt the candidate-JD matching pipeline from scratch, taking it from 0.72 to 0.91 NDCG@10, serving 50M+ queries per month. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage product companies and at larger scale, and I've spent enough time on both that I know which tradeoffs apply where. I believe most ranking problems are solved by careful feature engineering and rigorous eval, not by bigger models. Currently exploring my next move — looking for senior IC or tech-lead roles where I can own the intelligence layer end-to-end.","Indore, Madhya Pradesh",India,16.2,Senior Applied Scientist,Meta,10001+,Internet
93330,CAND_0093331,Kiara Menon,NLP Engineer | Applied ML | Building intelligent products,"Machine lea

In [15]:
profiles_df["current_title"].value_counts().head(30)

current_title
Business Analyst            5833
HR Manager                  5830
Mechanical Engineer         5791
Accountant                  5764
Project Manager             5754
Customer Support            5750
Operations Manager          5744
Content Writer              5727
Sales Executive             5713
Civil Engineer              5702
Graphic Designer            5689
Marketing Manager           5524
Software Engineer           3450
Full Stack Developer        2873
Cloud Engineer              2836
Java Developer              2809
.NET Developer              2788
DevOps Engineer             2787
Mobile Developer            2757
Frontend Engineer           2738
QA Engineer                 2682
Analytics Engineer           764
Data Engineer                744
Data Analyst                 728
Backend Engineer             704
Senior Data Engineer         687
Senior Software Engineer     653
ML Engineer                  167
AI Research Engineer         153
Data Scientist               

In [16]:
profiles_df["current_industry"].value_counts().head(30)

current_industry
IT Services             29881
Software                22417
Manufacturing           22305
Conglomerate             7571
Paper Products           7467
Fintech                  2808
Food Delivery            2514
E-commerce               1529
Consulting               1274
EdTech                    610
SaaS                      328
AI/ML                     278
AdTech                    172
Transportation            162
Insurance Tech            155
Gaming                    149
HealthTech                147
HealthTech AI              68
Conversational AI          62
AI Services                42
Voice AI                   31
Internet                   22
Media                       6
Consumer Electronics        2
Name: count, dtype: int64

In [17]:
profiles_df["country"].value_counts().head(30)

country
India        75113
USA           9978
Australia     2579
Canada        2506
UK            2472
Germany       2469
Singapore     2453
UAE           2430
Name: count, dtype: int64

In [18]:
profiles_df["current_company_size"].value_counts()


current_company_size
10001+        40464
1001-5000     18201
201-500       15096
51-200         7727
11-50          7568
501-1000       7525
5001-10000     3419
Name: count, dtype: int64

In [20]:
profiles_df["current_company"].value_counts().head(30)

current_company
Infosys              7590
Wayne Enterprises    7571
Wipro                7566
Initech              7528
Pied Piper           7500
Globex Inc           7492
Acme Corp            7490
Dunder Mifflin       7467
TCS                  7451
Hooli                7378
Stark Industries     7323
Swiggy               1288
Accenture            1274
Capgemini            1265
CRED                 1257
HCL                  1250
Razorpay             1246
Zomato               1226
Mindtree             1225
Cognizant            1213
Flipkart             1171
Tech Mahindra        1168
Mphasis              1153
Meesho                186
Nykaa                 172
InMobi                172
Zoho                  165
Vedantu               163
Freshworks            163
Paytm                 161
Name: count, dtype: int64

In [21]:
all_skills = []
for candidate in candidates:
    for skill in candidate["skills"]:
        all_skills.append(skill["name"])

skill_counts = (
    pd.Series(all_skills).value_counts()
)

skill_counts.head(50)

HTML                  12246
Databricks            12244
Redux                 12222
Terraform             12187
Angular               12173
Salesforce CRM        12157
Figma                 12157
Vue.js                12142
Sales                 12138
Accounting            12136
Agile                 12135
Kafka                 12114
Excel                 12109
BigQuery              12108
CI/CD                 12108
Project Management    12106
Airflow               12105
Flask                 12104
AWS                   12104
Scrum                 12083
Illustrator           12072
Kubernetes            12071
ETL                   12068
CSS                   12065
Docker                12062
Next.js               12058
Apache Beam           12054
Go                    12049
Java                  12049
TypeScript            12048
JavaScript            12047
dbt                   12046
REST APIs             12040
Spark                 12038
Marketing             12037
Tally               

In [27]:
skill_df = []
for candidate in candidates:
    for skill in candidate["skills"]:
        skill_df.append({
            "candidate_id" : candidate["candidate_id"],
            "skill_name" : skill["name"],
            "proficiency" : skill["proficiency"],
            "endorsements" : skill["endorsements"],
            "duration_months" : skill.get("duration_months", 0)
        })
skill_df = pd.DataFrame(skill_df)
skill_df.head()

,candidate_id,skill_name,proficiency,endorsements,duration_months
0,CAND_0000001,Tailwind,intermediate,3,13
1,CAND_0000001,NLP,advanced,37,26
2,CAND_0000001,Image Classification,advanced,7,40
3,CAND_0000001,Fine-tuning LLMs,advanced,21,36
4,CAND_0000001,Weights & Biases,intermediate,13,30


In [29]:
print(skill_df.columns)

Index(['candidate_id', 'skill_name', 'proficiency', 'endorsements',
       'duration_months'],
      dtype='object')


In [30]:
ai_keywords = [
    "Python",
    "NLP",
    "Embeddings",
    "Vector Database",
    "Milvus",
    "FAISS",
    "Qdrant",
    "Pinecone",
    "Weaviate",
    "Elasticsearch",
    "OpenSearch",
    "Sentence Transformers",
    "LoRA",
    "QLoRA",
    "PEFT",
    "Retrieval",
    "Ranking",
    "Recommendation Systems",
    "BM25"
]

skill_df[
    skill_df["skill_name"].str.contains(
        "|".join(ai_keywords),
        case=False,
        na=False
    )
].head(50)

,candidate_id,skill_name,proficiency,endorsements,duration_months
1,CAND_0000001,NLP,advanced,37,26
8,CAND_0000001,LoRA,intermediate,0,28
13,CAND_0000001,Milvus,advanced,40,35
83,CAND_0000010,Elasticsearch,intermediate,15,17
90,CAND_0000010,Python,intermediate,7,14
91,CAND_0000010,BM25,advanced,55,55
94,CAND_0000011,Recommendation Systems,advanced,5,40
124,CAND_0000014,FAISS,advanced,40,44
127,CAND_0000014,OpenSearch,advanced,47,59
140,CAND_0000015,Qdrant,intermediate,13,9


In [31]:
career_rows = []

for candidate in candidates:
    cid = candidate["candidate_id"]

    for job in candidate["career_history"]:
        career_rows.append({
            "candidate_id": cid,
            "company": job["company"],
            "title": job["title"],
            "industry": job["industry"],
            "company_size": job["company_size"],
            "start_date": job["start_date"],
            "end_date": job["end_date"],
            "duration_months": job["duration_months"],
            "is_current": job["is_current"],
            "description": job["description"]
        })

career_df = pd.DataFrame(career_rows)

career_df.head()

,candidate_id,company,title,industry,company_size,start_date,end_date,duration_months,is_current,description
0,CAND_0000001,Mindtree,Backend Engineer,IT Services,10001+,2024-03-08,None,27,True,"Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure."
1,CAND_0000001,Dunder Mifflin,Analytics Engineer,Paper Products,201-500,2019-07-03,2024-01-08,55,False,Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues — wrote most of the data quality checks that detect schema drift and unusual volume changes. The pipeline supports the analytics team and a few internal ML models.
2,CAND_0000002,Wipro,Operations Manager,IT Services,10001+,2022-11-14,None,43,True,Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise.
3,CAND_0000002,Wipro,Operations Manager,IT Services,10001+,2021-09-13,2022-11-07,14,False,"Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
4,CAND_0000002,Acme Corp,Marketing Manager,Manufacturing,201-500,2017-03-08,2021-08-14,54,False,"Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics — including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."


In [32]:
career_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300171 entries, 0 to 300170
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   candidate_id     300171 non-null  object
 1   company          300171 non-null  object
 2   title            300171 non-null  object
 3   industry         300171 non-null  object
 4   company_size     300171 non-null  object
 5   start_date       300171 non-null  object
 6   end_date         200171 non-null  object
 7   duration_months  300171 non-null  int64 
 8   is_current       300171 non-null  bool  
 9   description      300171 non-null  object
dtypes: bool(1), int64(1), object(8)
memory usage: 20.9+ MB


In [33]:
career_df["title"].value_counts().head(30)

title
Business Analyst            19042
Graphic Designer            19018
Project Manager             19016
Mechanical Engineer         18992
Accountant                  18955
Civil Engineer              18882
HR Manager                  18875
Customer Support            18842
Operations Manager          18799
Marketing Manager           18793
Content Writer              18786
Sales Executive             18695
Software Engineer            8081
Full Stack Developer         6832
Java Developer               6776
Cloud Engineer               6732
Mobile Developer             6722
.NET Developer               6707
QA Engineer                  6569
DevOps Engineer              6549
Frontend Engineer            6498
Data Engineer                1637
Analytics Engineer           1634
Data Analyst                 1555
Senior Data Engineer         1543
Backend Engineer             1526
Senior Software Engineer     1479
ML Engineer                   340
Junior ML Engineer            330
AI Resea

In [34]:
career_df["company"].value_counts().head(30)

company
Infosys              23722
Wipro                23682
Pied Piper           23614
Initech              23590
Wayne Enterprises    23556
Acme Corp            23546
Stark Industries     23524
Hooli                23509
TCS                  23483
Globex Inc           23471
Dunder Mifflin       23416
Swiggy                3019
Razorpay              2926
CRED                  2908
Capgemini             2895
HCL                   2894
Zomato                2883
Flipkart              2882
Mindtree              2879
Accenture             2871
Cognizant             2863
Tech Mahindra         2837
Mphasis               2822
Meesho                 384
Nykaa                  378
InMobi                 376
BYJU'S                 355
PolicyBazaar           355
Ola                    353
Zoho                   352
Name: count, dtype: int64

In [35]:
career_df["duration_months"].describe()

count    300171.000000
mean         28.236229
std          12.489886
min           6.000000
25%          18.000000
50%          27.000000
75%          38.000000
max         228.000000
Name: duration_months, dtype: float64